# dim_date

Kalenderdimensie gegenereerd uit `[MIN(order_ts), MAX(order_ts)]` via `SEQUENCE` + `EXPLODE`, zodat ook dagen zonder orders aanwezig zijn (geen gaten in tijdseries). `SHA2(full_date)` is de surrogate key.

Georkestreerd door `views/07_apply_views.ipynb`.

In [ ]:
%sql
CREATE WIDGET TEXT catalog DEFAULT "DEMO";
USE CATALOG ${catalog};

CREATE OR REPLACE VIEW DATAMART.dim_date AS
WITH date_bounds AS (
  SELECT
    MIN(CAST(order_ts AS DATE)) AS start_date,
    MAX(CAST(order_ts AS DATE)) AS end_date
  FROM INTEGRATION.order_header
),
dates AS (
  SELECT EXPLODE(SEQUENCE(start_date, end_date, INTERVAL 1 DAY)) AS full_date
  FROM date_bounds
)
SELECT
  SHA2(CAST(full_date AS STRING), 256)                       AS dim_date_key,
  full_date,
  YEAR(full_date)                                            AS year,
  QUARTER(full_date)                                         AS quarter,
  MONTH(full_date)                                           AS month,
  DATE_FORMAT(full_date, 'MMMM')                             AS month_name,
  DAY(full_date)                                             AS day,
  DAYOFWEEK(full_date)                                       AS day_of_week,
  DATE_FORMAT(full_date, 'EEEE')                             AS day_name,
  WEEKOFYEAR(full_date)                                      AS week_of_year,
  DAYOFWEEK(full_date) IN (1, 7)                             AS is_weekend,
  CAST(DATE_TRUNC('month',   full_date) AS DATE)             AS year_month_start,
  CAST(DATE_TRUNC('quarter', full_date) AS DATE)             AS year_quarter_start,
  CAST(DATE_TRUNC('year',    full_date) AS DATE)             AS year_start
FROM dates;